# ARIMA+GARCH Trading Strategy on the S&P500 Stock 

If you've followed our past series on [time series analysis](https://github.com/Auquan/Tutorials#time-series-analysis), you're now all familiar with the powerful ARIMA and GARCH models to model financial series.

In this article, we will apply the previous knowledge in a practical example to develop a trading strategy for the S&P500 

## Strategy Overview

The idea of this strategy is as below:

* For each day, $n$, use differenced logarithmic returns of S&P 500 for the previous $k$ days to fit an optimal ARIMA and GARCH model
* Use the combined model to make a prediction for the next day returns
* If the prediction is positive, go long the stock and if negative, short the stock at $n_{th}$ day's close
* If the prediction is the same as the previous day then do nothing

In this sheet we will work with $k=500$, but this is parameter that can be optimised in order to improve performance or reduce drawdown.

**Note: The backtest is doesnot take comission or slippage into account, hence the performance achieved in a real trading system would be lower than what you see here.**

## Strategy Implementation

In [11]:
import os
import sys

import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
import statsmodels.tsa.api as smt
import statsmodels.api as sm
import scipy.stats as scs
import statsmodels.stats as sms
from arch import arch_model

import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

In [12]:
# from backtester.dataSource.yahoo_data_source import YahooStockDataSource
# startDateStr = '2014/12/31'
# endDateStr = '2017/12/31'
# cachedFolderName = '/Users/dell/Auquan/auquantoolbox/yahooData/'
# dataSetId = 'testPairsTrading'
# instrumentIds = ['^GSPC']
# ds = YahooStockDataSource(cachedFolderName=cachedFolderName,
#                             dataSetId=dataSetId,
#                             instrumentIds=instrumentIds,
#                             startDateStr=startDateStr,
#                             endDateStr=endDateStr,
#                             event='history')
# data = ds.getBookDataByFeature()['adjClose']
# # log returns
# lrets = np.log(data/data.shift(1)).fillna(0)
# lrets.index = pd.DatetimeIndex(lrets.index.values,freq=lrets.index.inferred_freq)


In [23]:
from pathlib import Path

# Prefer local workspace data. If missing, ask before downloading.
local_candidates = [
    Path('../price-data/indexes/GSPC.csv'),
    Path('../price-data/SP500-1d.csv'),
    Path('../price-data/^GSPC-1d.csv'),
    Path('../price-data/indexes/SP500-1d.csv'),
    Path('../price-data/indexes/^GSPC-1d.csv'),
]

local_path = next((p for p in local_candidates if p.exists()), None)

if local_path is not None:
    odf = pd.read_csv(local_path)
    odf.columns = [c.strip().lower() for c in odf.columns]
    if 'date' in odf.columns:
        odf['dtime'] = pd.to_datetime(odf['date'])
    elif 'timestamp' in odf.columns:
        odf['dtime'] = pd.to_datetime(odf['timestamp'], unit='ms', errors='coerce')
    else:
        raise ValueError(f'Unsupported local file format: {local_path}')
    keep = [c for c in ['open', 'high', 'low', 'close', 'volume'] if c in odf.columns]
    odf = odf.dropna().set_index('dtime').sort_index()[keep]
    print(f"Data source: local file -> {local_path}")
else:
    print('No local S&P500 file found in ../price-data.')
    try:
        _confirm = input("Download S&P500 data from Yahoo Finance now? [y/N]: ").strip().lower()
    except EOFError:
        _confirm = ''

    if _confirm not in {'y', 'yes'}:
        raise RuntimeError(
            'Download not confirmed. Add a local file under ../price-data and rerun Cell 4.'
        )

    try:
        import yfinance as yf
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "yfinance is required for download mode. Install it in this notebook first (for example: %pip install yfinance)."
        ) from exc

    odf = (
        yf.Ticker('^GSPC')
        .history(period='max')
        .rename(columns={
            'Open': 'open',
            'High': 'high',
            'Low': 'low',
            'Close': 'close',
            'Volume': 'volume',
        })
        .dropna()
        .sort_index()
        .tail(2000)
    )
    odf.index.name = 'dtime'
    print('Data source: downloaded from Yahoo Finance (^GSPC)')

odf

Data source: local file -> ../price-data/indexes/GSPC.csv


,open,high,low,close,volume
dtime,,,,,
1960-01-04,59.910000,59.910000,59.910000,59.910000,3990000
1960-01-05,60.389999,60.389999,60.389999,60.389999,3710000
1960-01-06,60.130001,60.130001,60.130001,60.130001,3730000
1960-01-07,59.689999,59.689999,59.689999,59.689999,3310000
1960-01-08,59.500000,59.500000,59.500000,59.500000,3290000
...,...,...,...,...,...
2026-04-02,6512.609863,6601.910156,6474.939941,6582.689941,4740740000
2026-04-06,6587.660156,6618.129883,6579.720215,6611.830078,3906440000
2026-04-07,6601.930176,6618.259766,6534.549805,6616.850098,4555680000


In [24]:
lrets = odf.close.apply(np.log2).diff().fillna(0)
# lrets = np.log(data/data.shift(1)).fillna(0)
lrets.index = pd.DatetimeIndex(lrets.index.values,freq=lrets.index.inferred_freq)
lrets


1960-01-04    0.000000
1960-01-05    0.011513
1960-01-06   -0.006225
1960-01-07   -0.010596
1960-01-08   -0.004600
                ...   
2026-04-02    0.001616
2026-04-06    0.006372
2026-04-07    0.001095
2026-04-08    0.033742
2026-04-09    0.010871
Name: close, Length: 16677, dtype: float64

In [25]:
def tsplot(y, lags=None, figsize=(15, 10), style='bmh'):
    if not isinstance(y, pd.Series):
        y = pd.Series(y)
    with plt.style.context(style):    
        fig = plt.figure(figsize=figsize)
        #mpl.rcParams['font.family'] = 'Ubuntu Mono'
        layout = (3, 2)
        ts_ax = plt.subplot2grid(layout, (0, 0), colspan=2)
        acf_ax = plt.subplot2grid(layout, (1, 0))
        pacf_ax = plt.subplot2grid(layout, (1, 1))
        qq_ax = plt.subplot2grid(layout, (2, 0))
        pp_ax = plt.subplot2grid(layout, (2, 1))
        
        y.plot(ax=ts_ax)
        ts_ax.set_title('Time Series Analysis Plots')
        smt.graphics.plot_acf(y, lags=lags, ax=acf_ax, alpha=0.05)
        smt.graphics.plot_pacf(y, lags=lags, ax=pacf_ax, alpha=0.05)
        sm.qqplot(y, line='s', ax=qq_ax)
        qq_ax.set_title('QQ Plot')        
        scs.probplot(y, sparams=(y.mean(), y.std()), plot=pp_ax)

        plt.tight_layout()
    return

In [26]:
windowLength = 150
foreLength = len(lrets) - windowLength

We also use [previously defined functions](https://github.com/Auquan/Tutorials/blob/master/Time%20Series%20Analysis%20-%204.ipynb) to find best ARIMA model

In [27]:
def _get_best_model(TS):
    best_aic = np.inf 
    best_order = None
    best_mdl = None

    pq_rng = range(1,4) # [1,2,3]
    for i in pq_rng:
            for j in pq_rng:
                # try:
                    tmp_mdl = smt.ARIMA(TS, order=(i,0,j), trend='ct').fit()
                    tmp_aic = tmp_mdl.aic
                    if tmp_aic < best_aic:
                        best_aic = tmp_aic
                        best_order = (i, 0, j)
                        best_mdl = tmp_mdl
                # except: continue
    print('aic: %6.5f | order: %s'%(best_aic, best_order))                    
    return best_aic, best_order, best_mdl



In [28]:
d = 100
# windowLength = 150
TS = lrets[(1+d):(windowLength+d)].values
# tmp_mdl = smt.ARIMA(TS, order=(3,0,3), trend='ct').fit()
# tmp_mdl.aic
   # Find the best ARIMA fit (we set differencing to 0 since we've already differenced the series once)
# res_tup = _get_best_model(TS)
# res_tup
# TS.values
# res = mod.fit()
# smt.ARIMA(TS, order=(3,0,3),).fit().aic

At this stage we need to loop through every day in the trading data and fit an appropriate ARIMA and GARCH model to the rolling window of length $k=500$. Given that we try 32 separate ARIMA fits and fit a GARCH model, for each day, the indicator can take a long time to generate.

In [ ]:
from tqdm.auto import tqdm

signal = 0*lrets[-foreLength:]
for d in tqdm(range(foreLength), desc="Rolling ARIMA+GARCH"):
    # create a rolling window by selecting the values between 1+d and k+d of S&P500 returns
    TS = (10*lrets)[(1+d):(windowLength+d)] 
    # Find the best ARIMA fit (we set differencing to 0 since we've already differenced the series once)
    res_tup = _get_best_model(TS)
    order = res_tup[1]
    model = res_tup[2]
    #now that we have our ARIMA fit, we feed this to GARCH model
    p_ = order[0]
    o_ = order[1]
    q_ = order[2]

    am = arch_model(model.resid, p=p_, o=o_, q=q_, dist='StudentsT')
    res = am.fit(update_freq=5, disp='off')
    out = res.forecast(horizon=1, start=None, align='origin')
    signal.iloc[d] = np.sign(out.mean['h.1'].iloc[-1])

/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check m

aic: -292.79875 | order: (1, 0, 1)


/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.007676. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. fo

aic: -293.24454 | order: (1, 0, 1)


/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated 

aic: -293.26216 | order: (1, 0, 1)


... (12 outputs left out)

/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.007434. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. fo

aic: -297.61068 | order: (1, 0, 1)


/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.00743. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/grenada/miniconda3/envs/py313/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. for

KeyboardInterrupt: 

## Strategy Results

Now that we have generated our signals, we need to compare its performance to "Buy & Hold".


In [ ]:
rets = pd.DataFrame(index = signal.index, columns=['Buy and Hold', 'Strategy'])
rets['Buy and Hold'] = lrets[-foreLength:]
rets['Strategy'] = signal
rets['Strategy'] = rets['Strategy']*rets['Buy and Hold']
eqCurves['Buy and Hold']=rets['Buy and Hold'].cumsum()+1
eqCurves['Strategy'] = rets['Strategy'].cumsum()+1
eqCurves['Strategy'].plot(figsize=(10,8))
eqCurves['Buy and Hold'].plot()
plt.legend()
plt.show()